# A1组 6.2–6.3 习题参考解答代码

> 配套文件：题目 `A1组-习题.md`，解答 `A1组-习题参考解答.md`。本 Notebook 把 4 个主脚本与 1 个残差检查合并在一起，按单元顺序执行即可复现解答中的数值与图片。

- **代码载体：** 全部代码在本 Notebook 内，不导入外部 `.py`。
- **相对路径：** 从本文件所在的提交包根目录启动 Jupyter；图写入 `图片/`，结果 JSON 写入 `数据/`，不新建其他文件夹。
- **环境：** Python 3.12.x（原始运行 Python 3.12.10）；NumPy、PyTorch CPU、Matplotlib；不依赖 scipy（题目 6.2-2 的一维极小化用等价的黄金分割搜索实现）。
- **运行顺序：** 6.2-1 → 6.2-2 → 6.3-1 → 6.3-2 → 附录残差检查。完整运行约数分钟；各单元会打印本单元耗时。
- **说明：** 训练使用固定随机种子；若更换硬件或 PyTorch 版本，末位数值可能有小幅波动。解答中的结果为本次交付前的实际运行值。

### 运行说明

本 Notebook 的 4 个训练单元使用固定种子；在本机 8 核 CPU 上完整串行运行约 30–60 分钟（小张量、float64 的逐点优化占主要时间）。为避免 Notebook 携带超长日志，四个训练单元的输出取自**同一代码的原始运行结果**，并已保存为 `数据/A1组-6.2-习题1结果.json` 等四个文件；附录残差检查在本环境实际执行。代码已加入 `torch.set_num_threads(1)`，用于降低小张量在线程间同步的开销，不改变算法。



## 题 6.2-1 参数缩放对称与观测等价

对应题面 (1)–(4)。本单元完成：解析解与边界通量；固定 $(k,r)$ 的两组 PINN 正问题；从两种初值出发的反演；损失景观扫描。重点结论：$(k,r)=(1,4)$ 与 $(0.25,1)$ 给出同一条解，仅靠内部解观测无法区分；边界通量 $q(0),q(1)=-ku'$ 可以区分。

输出：`数据/A1组-6.2-习题1结果.json`；图见 `图片/A1组-6.2-习题1*.png`。

In [1]:
import os, json, math, time
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
np.random.seed(0)
torch.manual_seed(0)

ROOT = "数据"
FIG = "图片"
assert os.path.isdir(FIG), "缺少 图片/ 目录；请在提交包根目录下启动 Jupyter"

m_true = 2.0
kA, rA = 1.0, 4.0
kB, rB = 0.25, 1.0

def u_exact(x, k, r):
    m = math.sqrt(r/k)
    if abs(m) < 1e-12:
        return x
    return np.sinh(m*x)/np.sinh(m)

x_obs = (np.arange(21)+0.5)/21.0
u_obs = u_exact(x_obs, kA, rA)
sigma = 0.005

def q0(k, r):
    m = math.sqrt(r/k)
    return -k*m/math.sinh(m)

def q1(k, r):
    m = math.sqrt(r/k)
    return -k*m/math.tanh(m)

class MLP(nn.Module):
    def __init__(self, width=64, depth=3):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
        for p in self.parameters():
            if p.ndim == 2:
                nn.init.xavier_normal_(p)
    def forward(self, x):
        return self.net(x)

def u_ansatz(model, x):
    # hard constraint u(0)=0, u(1)=1
    return x + x*(1.0-x)*model(x)

def pde_residual(model, x, k, r):
    x = x.clone().requires_grad_(True)
    u = u_ansatz(model, x)
    u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
    return -k*u_xx + r*u

def train_fixed(k, r, seed=0, iters=8000):
    torch.manual_seed(seed)
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    x_obs_t = torch.tensor(x_obs.reshape(-1,1))
    u_obs_t = torch.tensor(u_obs.reshape(-1,1))
    x_col = torch.linspace(0,1,400).reshape(-1,1)
    x_bc = torch.tensor([[0.0],[1.0]])
    hist = []
    for it in range(iters):
        opt.zero_grad()
        u_pred = u_ansatz(model, x_obs_t)
        loss_d = torch.mean((u_pred-u_obs_t)**2)
        res = pde_residual(model, x_col, k, r)
        loss_p = torch.mean(res**2)
        u_bc = u_ansatz(model, x_bc)
        loss_b = torch.mean((u_bc - torch.tensor([[0.0],[1.0]]))**2)
        loss = loss_d + 1e-2*loss_p + 1e2*loss_b
        loss.backward()
        opt.step()
        if it % 100 == 0:
            hist.append([it, float(loss_d), float(loss_p), float(loss_b)])
    xg = np.linspace(0,1,201)
    with torch.no_grad():
        u_pred = u_ansatz(model, torch.tensor(xg.reshape(-1,1))).numpy().ravel()
    ue = u_exact(xg, k, r)
    rel = np.linalg.norm(u_pred-ue)/np.linalg.norm(ue)
    # data metrics
    with torch.no_grad():
        u_at_obs = u_ansatz(model, torch.tensor(x_obs.reshape(-1,1))).numpy().ravel()
        bc = u_ansatz(model, torch.tensor([[0.0],[1.0]])).numpy().ravel()
    res = pde_residual(model, torch.linspace(0,1,240).reshape(-1,1), k, r).detach().numpy().ravel()
    return dict(model=model, hist=hist, xg=xg, u_pred=u_pred, ue=ue, rel=float(rel),
                data_rmse=float(np.sqrt(np.mean((u_at_obs-u_obs)**2))),
                pde_rms=float(np.sqrt(np.mean(res**2))),
                bc_max=float(np.max(np.abs(bc-np.array([0.0,1.0])))))

def train_inverse(init_k, init_r, seed=0, iters=8000):
    torch.manual_seed(seed)
    model = MLP()
    logk = nn.Parameter(torch.tensor(math.log(init_k)))
    logr = nn.Parameter(torch.tensor(math.log(init_r)))
    opt = torch.optim.Adam(list(model.parameters())+[logk, logr], lr=2e-3)
    x_obs_t = torch.tensor(x_obs.reshape(-1,1))
    u_obs_t = torch.tensor(u_obs.reshape(-1,1))
    x_col = torch.linspace(0,1,400).reshape(-1,1)
    x_bc = torch.tensor([[0.0],[1.0]])
    hist = []
    for it in range(iters):
        opt.zero_grad()
        k = torch.exp(logk); r = torch.exp(logr)
        u_pred = u_ansatz(model, x_obs_t)
        loss_d = torch.mean((u_pred-u_obs_t)**2)
        res = pde_residual(model, x_col, k, r)
        loss_p = torch.mean(res**2)
        u_bc = u_ansatz(model, x_bc)
        loss_b = torch.mean((u_bc - torch.tensor([[0.0],[1.0]]))**2)
        loss = loss_d + 1e-2*loss_p + 1e2*loss_b
        loss.backward()
        opt.step()
        if it % 100 == 0:
            hist.append([it, float(loss_d), float(loss_p), float(loss_b), float(torch.exp(logk)), float(torch.exp(logr))])
    kk, rr = float(torch.exp(logk)), float(torch.exp(logr))
    with torch.no_grad():
        u_at_obs = u_ansatz(model, torch.tensor(x_obs.reshape(-1,1))).numpy().ravel()
    return dict(k=kk, r=rr, q0=q0(kk,rr), data_rmse=float(np.sqrt(np.mean((u_at_obs-u_obs)**2))), hist=hist)

t0 = time.time()
fixed = {}
for name,(k,r) in {"A":(kA,rA), "B":(kB,rB)}.items():
    fixed[name] = train_fixed(k,r,seed=0,iters=8000)
    print(name, fixed[name]["rel"], fixed[name]["data_rmse"], fixed[name]["pde_rms"], fixed[name]["bc_max"])

inv1 = train_inverse(1.0,4.0,seed=1,iters=8000)
inv2 = train_inverse(0.5,1.0,seed=2,iters=8000)
print("inv1", inv1["k"], inv1["r"], inv1["q0"], inv1["data_rmse"])
print("inv2", inv2["k"], inv2["r"], inv2["q0"], inv2["data_rmse"])

# loss landscape from analytic u
kg = np.linspace(0.1, 2.0, 161)
rg = np.linspace(0.2, 8.0, 161)
K, R = np.meshgrid(kg, rg)
L = np.zeros_like(K)
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        uu = u_exact(x_obs, K[i,j], R[i,j])
        L[i,j] = np.mean((uu-u_obs)**2)

# figures
xg = fixed["A"]["xg"]
plt.figure(figsize=(6.4,4.2))
plt.plot(xg, fixed["A"]["ue"], 'k-', lw=2.2, label='analytic A/B')
plt.plot(xg, fixed["A"]["u_pred"], 'C0--', lw=1.6, label='PINN A')
plt.plot(xg, fixed["B"]["u_pred"], 'C1:', lw=1.6, label='PINN B')
plt.xlabel('x'); plt.ylabel('u(x)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1双参数场.png"), dpi=180); plt.close()

plt.figure(figsize=(6.4,4.6))
cs = plt.contourf(K, R, np.log10(L+1e-30), levels=40, cmap='viridis')
plt.colorbar(cs, label=r'$\log_{10}$ data MSE')
plt.plot([kA,kB],[rA,rB],'o', color='red', ms=7, label='A and B')
plt.plot(kg, 4*kg, 'w--', lw=1.2, label=r'$r=4k$')
plt.xlabel('k'); plt.ylabel('r'); plt.legend(loc='upper left'); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1损失山谷.png"), dpi=180); plt.close()

plt.figure(figsize=(6.0,4.0))
xx = np.arange(2); w=0.35
plt.bar(xx-w/2, [q0(kA,rA), q1(kA,rA)], w, label='A (1,4)')
plt.bar(xx+w/2, [q0(kB,rB), q1(kB,rB)], w, label='B (0.25,1)')
plt.xticks(xx, ['q(0)','q(1)']); plt.ylabel('flux'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1边界通量.png"), dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
for name in ['A','B']:
    h = np.array(fixed[name]["hist"])
    plt.semilogy(h[:,0], h[:,1]+1e-16, label=f'data {name}')
    plt.semilogy(h[:,0], h[:,2]+1e-16, '--', label=f'PDE {name}')
plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1训练曲线.png"), dpi=180); plt.close()

res = {
  "analytic": {"max_abs_uA_uB": float(np.max(np.abs(u_exact(xg,kA,rA)-u_exact(xg,kB,rB)))),
               "q0_A": q0(kA,rA), "q0_B": q0(kB,rB), "q1_A": q1(kA,rA), "q1_B": q1(kB,rB),
               "q0_diff": q0(kA,rA)-q0(kB,rB), "q1_diff": q1(kA,rA)-q1(kB,rB)},
  "fixed": {name: {kk: fixed[name][kk] for kk in ["rel","data_rmse","pde_rms","bc_max"]} for name in fixed},
  "inverse": {"init_A": {kk: inv1[kk] for kk in ["k","r","q0","data_rmse"]},
               "init_B": {kk: inv2[kk] for kk in ["k","r","q0","data_rmse"]},
               "q0_A_true": q0(kA,rA), "q0_B_true": q0(kB,rB)},
  "time_sec": time.time()-t0,
  "loss_min": float(L.min()), "loss_max": float(L.max())
}
with open(os.path.join(ROOT,"A1组-6.2-习题1结果.json"),"w",encoding="utf-8") as f:
    json.dump(res,f,ensure_ascii=False,indent=2)
print(json.dumps(res,ensure_ascii=False,indent=2))



{
  "analytic": {
    "max_abs_uA_uB": 0.0,
    "q0_A": -0.5514411295435665,
    "q0_B": -0.13786028238589162,
    "q1_A": -2.0746294414550963,
    "q1_B": -0.5186573603637741,
    "q0_diff": -0.41358084715767485,
    "q1_diff": -1.5559720810913222
  },
  "fixed": {
    "A": {
      "rel": 1.6070779042220406e-05,
      "data_rmse": 7.602229316705281e-06,
      "pde_rms": 0.0014882022081310457,
      "bc_max": 0.0
    },
    "B": {
      "rel": 3.0962858297149044e-05,
      "data_rmse": 1.464758917867401e-05,
      "pde_rms": 0.0006155801817453438,
      "bc_max": 0.0
    }
  },
  "inverse": {
    "init_A": {
      "k": 0.9163447396150999,
      "r": 3.665306941914612,
      "q0": -0.5053155128815252,
      "data_rmse": 9.348695870114462e-06
    },
    "init_B": {
      "k": 0.35520620153922755,
      "r": 1.4208002107396203,
      "q0": -0.19587713089869624,
      "data_rmse": 7.947577845386675e-06
    },
    "q0_A_true": -0.5514411295435665,
    "q0_B_true": -0.13786028238589162
  },


### 6.2-1 结果图

![6.2-1 双参数场](图片/A1组-6.2-习题1双参数场.png)

![6.2-1 损失山谷](图片/A1组-6.2-习题1损失山谷.png)

![6.2-1 边界通量](图片/A1组-6.2-习题1边界通量.png)

![6.2-1 训练曲线](图片/A1组-6.2-习题1训练曲线.png)

## 题 6.2-2 正问题与反问题的损失及观测算子

对应题面 (1)–(4)。本单元完成：正问题 PINN；反问题只用内部数据估计 $a$；仅边界点布局的 $\chi^2(a)$ 平坦性；FIM 与 20 次噪声实现的标准差。重点结论：正问题没有数据项仍可解；反问题靠数据项识别 $a$，仅边界点对 $a$ 为零信息。

输出：`数据/A1组-6.2-习题2结果.json`；图见 `图片/A1组-6.2-习题2*.png`。

In [2]:
import os, json, math, time
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
# 本机交付环境未安装 scipy；用有界黄金分割搜索实现等价的一维极小化。
import math as _math

class _OptResult:
    pass

def minimize_scalar(fun, bounds=(0.0, 1.0), method="bounded", options=None, **_):
    """有界一维极小化（黄金分割），对应 scipy.optimize.minimize_scalar(method='bounded')。"""
    a, b = float(bounds[0]), float(bounds[1])
    tol = float((options or {}).get("xatol", 1e-10))
    gr = (_math.sqrt(5.0) - 1.0) / 2.0
    c, d = b - gr * (b - a), a + gr * (b - a)
    fc, fd = fun(c), fun(d)
    for _ in range(200):
        if b - a <= tol:
            break
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - gr * (b - a)
            fc = fun(c)
        else:
            a, c, fc = c, d, fd
            d = a + gr * (b - a)
            fd = fun(d)
    x = 0.5 * (a + b)
    out = _OptResult()
    out.x = x
    out.fun = float(fun(x))
    out.success = True
    return out


torch.set_default_dtype(torch.float64)
np.random.seed(1); torch.manual_seed(1)
ROOT = "数据"
FIG = "图片"
assert os.path.isdir(FIG), "缺少 图片/ 目录；请在提交包根目录下启动 Jupyter"

pi2=math.pi**2
def u_true(a,x):
    return (pi2+4)/(pi2+a)*np.sin(np.pi*x)

x_int=np.arange(1,9)/9.0
sigma=0.01
u_obs=u_true(4.0,x_int)+sigma*np.random.randn(8)

class MLP(nn.Module):
    def __init__(self,width=64,depth=3):
        super().__init__()
        layers=[nn.Linear(1,width),nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width,width),nn.Tanh()]
        layers += [nn.Linear(width,1)]
        self.net=nn.Sequential(*layers)
        for p in self.parameters():
            if p.ndim==2: nn.init.xavier_normal_(p)
    def forward(self,x): return self.net(x)

def ansatz(model,x):
    return x*(1.0-x)*model(x)

def pde_res(model,x,a):
    x=x.clone().requires_grad_(True)
    u=ansatz(model,x)
    ux=torch.autograd.grad(u.sum(),x,create_graph=True)[0]
    uxx=torch.autograd.grad(ux.sum(),x,create_graph=True)[0]
    f=(pi2+4)*torch.sin(math.pi*x)
    return -uxx+a*u-f

def train_forward(seed=0,iters=6000):
    torch.manual_seed(seed)
    model=MLP()
    opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    xcol=torch.linspace(0,1,300).reshape(-1,1)
    hist=[]
    for it in range(iters):
        opt.zero_grad()
        res=pde_res(model,xcol,4.0)
        loss=torch.mean(res**2)
        loss.backward(); opt.step()
        if it%200==0: hist.append([it,float(loss.detach())])
    xg=np.linspace(0,1,201)
    with torch.no_grad():
        up=ansatz(model,torch.tensor(xg.reshape(-1,1))).numpy().ravel()
    ue=np.sin(math.pi*xg)
    res=pde_res(model,torch.linspace(0,1,300).reshape(-1,1),4.0).detach().numpy().ravel()
    return dict(model=model,hist=hist,xg=xg,up=up,ue=ue,
                rel=float(np.linalg.norm(up-ue)/np.linalg.norm(ue)),
                pde_rms=float(np.sqrt(np.mean(res**2))),
                bc_max=float(max(abs(up[0]),abs(up[-1]))))

def train_inverse(seed=0,iters=6000):
    torch.manual_seed(seed)
    model=MLP()
    loga=nn.Parameter(torch.tensor(math.log(2.0)))
    opt=torch.optim.Adam(list(model.parameters())+[loga],lr=1e-3)
    xobs_t=torch.tensor(x_int.reshape(-1,1)); uobs_t=torch.tensor(u_obs.reshape(-1,1))
    xcol=torch.linspace(0,1,300).reshape(-1,1)
    hist=[]
    for it in range(iters):
        opt.zero_grad()
        a=torch.exp(loga)
        up=ansatz(model,xobs_t)
        loss_d=torch.mean((up-uobs_t)**2)/sigma**2
        res=pde_res(model,xcol,a)
        loss_p=torch.mean(res**2)
        loss=loss_d+1e-2*loss_p
        loss.backward(); opt.step()
        if it%200==0: hist.append([it,float(loss_d.detach()),float(loss_p.detach()),float(a.detach())])
    a=float(torch.exp(loga))
    with torch.no_grad():
        up=ansatz(model,xobs_t).numpy().ravel()
    return dict(a=a,data_rmse=float(np.sqrt(np.mean((up-u_obs)**2))),hist=hist)

t0=time.time()
fwd=train_forward(seed=0,iters=6000)
inv=train_inverse(seed=1,iters=6000)
print("forward",fwd["rel"],fwd["pde_rms"],fwd["bc_max"])
print("inverse",inv["a"],inv["data_rmse"])

# chi2 scans
a_grid=np.linspace(0.01,10,501)
chi_int=np.zeros_like(a_grid)
for i,a in enumerate(a_grid):
    chi_int[i]=np.sum((u_true(a,x_int)-u_obs)**2)/sigma**2
# boundary layout: u_a(0)=u_a(1)=0, observations zero (no noise here)
chi_bnd=np.zeros_like(a_grid)  # constant zero
# with noise on boundary it is still constant; report normalized constant
eps_b=np.array([0.0,0.0])
chi_bnd=np.full_like(a_grid, np.sum(eps_b**2)/sigma**2)

# FIM at a=4
du_da=-(pi2+4)/(pi2+4.0)**2*np.sin(np.pi*x_int)
I=float(np.sum(du_da**2)/sigma**2)
std_fim=1/math.sqrt(I)
# 4-point design
x4=np.arange(1,5)/5.0
du_da4=-(pi2+4)/(pi2+4.0)**2*np.sin(np.pi*x4)
I4=float(np.sum(du_da4**2)/sigma**2)
std4=1/math.sqrt(I4)

# 20 noise realizations, analytic least squares via scalar minimization
ests=[]; rng=np.random.default_rng(123)
for k in range(20):
    uo=u_true(4.0,x_int)+sigma*rng.standard_normal(8)
    fun=lambda a: np.sum((u_true(a,x_int)-uo)**2)/sigma**2
    r=minimize_scalar(fun,bounds=(0.01,10),method='bounded',options={'xatol':1e-10})
    ests.append(r.x)
ests=np.array(ests)

# figures
plt.figure(figsize=(6.4,4.2))
plt.plot(fwd["xg"],fwd["ue"],'k-',lw=2.2,label='analytic')
plt.plot(fwd["xg"],fwd["up"],'C0--',lw=1.6,label='PINN forward')
plt.xlabel('x'); plt.ylabel('u(x)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题2正问题解.png"),dpi=180); plt.close()

fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].semilogy(a_grid,chi_int+1e-12,'C0-')
ax[0].axvline(4,color='k',ls='--',lw=1)
ax[0].set_xlabel('a'); ax[0].set_ylabel(r'$\chi^2(a)$ (interior)'); ax[0].set_title('8 interior sensors')
ax[1].plot(a_grid,chi_bnd,'C1-')
ax[1].set_xlabel('a'); ax[1].set_ylabel(r'$\chi^2(a)$ (boundary)'); ax[1].set_title('boundary sensors only')
ax[1].set_ylim(-0.1,0.1)
plt.tight_layout(); plt.savefig(os.path.join(FIG,"A1组-6.2-习题2卡方曲线.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.2))
xx=np.linspace(0,1,201)
for a in [2,4,6]:
    plt.plot(xx,u_true(a,xx),label=f'a={a}')
plt.plot(x_int,u_obs,'ko',ms=4,label='observations')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题2不同a的解.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
plt.plot(np.array(fwd["hist"])[:,0],np.array(fwd["hist"])[:,1]+1e-16,label='forward PDE loss')
plt.yscale('log'); plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题2训练曲线.png"),dpi=180); plt.close()

res={
 "forward":{"rel":fwd["rel"],"pde_rms":fwd["pde_rms"],"bc_max":fwd["bc_max"]},
 "inverse_pinn":{"a":inv["a"],"data_rmse":inv["data_rmse"]},
 "fim":{"I":I,"std":std_fim,"I4":I4,"std4":std4},
 "noise20":{"median_abs_err":float(np.median(np.abs(ests-4))),"mean":float(np.mean(ests)),"std":float(np.std(ests,ddof=1)),"min":float(ests.min()),"max":float(ests.max())},
 "chi2":{"a_at_min":float(a_grid[np.argmin(chi_int)]),"min":float(chi_int.min()),"bnd_variation":float(chi_bnd.max()-chi_bnd.min())},
 "time_sec":time.time()-t0
}
with open(os.path.join(ROOT,"A1组-6.2-习题2结果.json"),"w",encoding="utf-8") as f:
    json.dump(res,f,ensure_ascii=False,indent=2)
print(json.dumps(res,ensure_ascii=False,indent=2))


{
  "forward": {
    "rel": 8.926606561214939e-06,
    "pde_rms": 0.002270142186283181,
    "bc_max": 0.0
  },
  "inverse_pinn": {
    "a": 3.981700434770421,
    "data_rmse": 0.008823954614976783
  },
  "fim": {
    "I": 233.9291623986534,
    "std": 0.06538194216256712,
    "I4": 129.9606457770297,
    "std4": 0.08771908030587582
  },
  "noise20": {
    "median_abs_err": 0.045842291800404,
    "mean": 3.974255917983828,
    "std": 0.05068896973411544,
    "min": 3.8530356181111967,
    "max": 4.069620144729059
  },
  "chi2": {
    "a_at_min": 4.04596,
    "min": 13.525975088585357,
    "bnd_variation": 0.0
  },
  "time_sec": 89.37853455543518
}


### 6.2-2 结果图

![6.2-2 正问题解](图片/A1组-6.2-习题2正问题解.png)

![6.2-2 卡方曲线](图片/A1组-6.2-习题2卡方曲线.png)

![6.2-2 不同 a 的解](图片/A1组-6.2-习题2不同a的解.png)

![6.2-2 训练曲线](图片/A1组-6.2-习题2训练曲线.png)

## 题 6.3-1 Darcy 算子的分布内/分布外与泄漏诊断

对应题面 (1)–(4)。本单元完成：160 个 Darcy 样本；按函数划分与随机划分两种协议；“同函数最近邻”泄漏诊断；两种划分下的训练与 MSE 对比。重点结论：按点随机划分会让同函数的不同坐标同时进入训练与测试，造成泄漏；按函数划分才能检验算子泛化。

输出：`数据/A1组-6.3-习题1结果.json`；图见 `图片/A1组-6.3-习题1*.png`。

In [3]:
import os, json, math, time
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)
np.random.seed(0); torch.manual_seed(0)
ROOT = "数据"
FIG = "图片"
assert os.path.isdir(FIG), "缺少 图片/ 目录；请在提交包根目录下启动 Jupyter"

N=160; Nx=64
x=np.linspace(0,1,Nx); h=x[1]-x[0]
rng=np.random.default_rng(2026)
kmod=np.arange(1,13)
xi=rng.standard_normal((N,12)); eta=rng.standard_normal((N,12))
loga=np.zeros((N,Nx))
for k in range(12):
    loga += (0.8/(k+1))*(xi[:,k,None]*np.cos(2*np.pi*(k+1)*x)[None,:]
                         + eta[:,k,None]*np.sin(2*np.pi*(k+1)*x)[None,:])
loga=np.clip(loga,-1.5,1.5)
a=np.exp(loga)

# finite-volume Darcy solve
u=np.zeros((N,Nx))
for n in range(N):
    af=2*a[n,:-1]*a[n,1:]/(a[n,:-1]+a[n,1:])  # length Nx-1
    m=Nx-2
    A=np.zeros((m,m)); rhs=np.full(m,h*h)
    for j in range(1,Nx-1):
        row=j-1
        left=af[j-1]; right=af[j]
        if row-1>=0: A[row,row-1]=-left
        A[row,row]=left+right
        if row+1<m: A[row,row+1]=-right
    u[n,1:-1]=np.linalg.solve(A,rhs)

# splits
train_ids=np.arange(0,100); val_ids=np.arange(100,120)
leak_ids=np.arange(120,140); unseen_ids=np.arange(140,160)
# S_point: pool train+leak, random 80/20
pool=np.concatenate([train_ids,leak_ids])
rng2=np.random.default_rng(7)
mask_train=np.zeros((len(pool),Nx),dtype=bool)
for i,n in enumerate(pool):
    idx=rng2.permutation(Nx)
    ntr=int(round(0.8*Nx))
    mask_train[i,idx[:ntr]]=True
mask_test=~mask_train
# ensure each function has train/test (by construction)

class FNO1d(nn.Module):
    def __init__(self,width=32,modes=16,layers=4):
        super().__init__()
        self.width=width; self.modes=modes
        self.lift=nn.Conv1d(1,width,1)
        self.skips=nn.ModuleList([nn.Conv1d(width,width,1) for _ in range(layers)])
        self.weights=nn.ParameterList([
            nn.Parameter(torch.randn(width,width,modes,2)*0.02) for _ in range(layers)
        ])
        self.acts=nn.ModuleList([nn.GELU() for _ in range(layers)])
        self.proj=nn.Conv1d(width,1,1)
    def forward(self,z):
        # z: [B,Nx]
        z=z[:,None,:]
        z=self.lift(z)
        for w,skip,act in zip(self.weights,self.skips,self.acts):
            zf=torch.fft.rfft(z,dim=-1)
            wc=torch.complex(w[...,0],w[...,1])
            out=torch.einsum('bix,iox->box',zf[:,:,:self.modes],wc)
            zf2=torch.zeros_like(zf)
            zf2[:,:,:self.modes]=out
            z=torch.fft.irfft(zf2,n=z.shape[-1],dim=-1)
            z=act(z+skip(z))
        return self.proj(z)[:,0,:]

def train_model(mode, seed=0, epochs=350):
    torch.manual_seed(seed)
    model=FNO1d()
    opt=torch.optim.Adam(model.parameters(),lr=2e-3,weight_decay=1e-6)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    # tensors
    if mode=='point':
        ids=pool; z=torch.tensor(loga[ids],dtype=torch.float32); y=torch.tensor(u[ids],dtype=torch.float32)
        mt=torch.tensor(mask_train); 
    else:
        ids=train_ids; z=torch.tensor(loga[ids],dtype=torch.float32); y=torch.tensor(u[ids],dtype=torch.float32)
        mt=None
    uscale=float(y.std())
    yn=y/uscale
    hist=[]
    for ep in range(epochs):
        perm=torch.randperm(len(ids))
        tot=0.0; nb=0
        for s in range(0,len(ids),20):
            idx=perm[s:s+20]
            opt.zero_grad()
            pred=model(z[idx])
            if mode=='point':
                mm=mt[idx]
                loss=((pred-yn[idx])**2)[mm].mean()
            else:
                loss=((pred-yn[idx])**2).mean()
            loss.backward(); opt.step()
            tot+=float(loss.detach()); nb+=1
        sched.step()
        if ep%10==0: hist.append([ep,tot/nb])
    return model,uscale,hist

def predict(model,uscale,ids):
    with torch.no_grad():
        return model(torch.tensor(loga[ids],dtype=torch.float32)).numpy()*uscale

t0=time.time()
mL,sL,hL=train_model('point',seed=0,epochs=350)
mG,sG,hG=train_model('func',seed=0,epochs=350)
print('trained',time.time()-t0)

# evaluate
# L on leak held-out points
predL=predict(mL,sL,leak_ids)
maskL_test=mask_test[np.isin(pool,leak_ids)]
# mask_test rows correspond to pool order; leak_ids are last 20 rows of pool
mse_leak=float(np.mean((predL[maskL_test]-u[leak_ids][maskL_test])**2))
rel_leak=float(np.linalg.norm(predL[maskL_test]-u[leak_ids][maskL_test])/np.linalg.norm(u[leak_ids][maskL_test]))
# L on unseen full
predL_un=predict(mL,sL,unseen_ids)
mse_unseenL=float(np.mean((predL_un-u[unseen_ids])**2))
rel_unseenL=float(np.linalg.norm(predL_un-u[unseen_ids])/np.linalg.norm(u[unseen_ids]))
# G on leak/unseen
predG_leak=predict(mG,sG,leak_ids)
mse_leakG=float(np.mean((predG_leak-u[leak_ids])**2))
rel_leakG=float(np.linalg.norm(predG_leak-u[leak_ids])/np.linalg.norm(u[leak_ids]))
predG_un=predict(mG,sG,unseen_ids)
mse_unseenG=float(np.mean((predG_un-u[unseen_ids])**2))
rel_unseenG=float(np.linalg.norm(predG_un-u[unseen_ids])/np.linalg.norm(u[unseen_ids]))

# audit
train_fid=pool[mask_train.any(axis=1)]
test_fid=pool[mask_test.any(axis=1)]
leak_rate=float(np.mean(np.isin(test_fid,train_fid)))
# nearest same-function train distance
same_d=[]
for i,n in enumerate(pool):
    tr_idx=np.where(mask_train[i])[0]
    te_idx=np.where(mask_test[i])[0]
    if len(te_idx)==0: continue
    for j in te_idx:
        same_d.append(np.min(np.abs(tr_idx-j)))
same_d=np.array(same_d)
frac2=float(np.mean(same_d<=2))
# 1-NN baseline
err_same=[]; err_cross=[]
# cross baseline: nearest other function at same x_j
for i,n in enumerate(pool):
    te_idx=np.where(mask_test[i])[0]
    tr_idx=np.where(mask_train[i])[0]
    for j in te_idx:
        jj=tr_idx[np.argmin(np.abs(tr_idx-j))]
        err_same.append((u[n,j]-u[n,jj])**2)
        other=[m for m in pool if m!=n]
        # nearest in function index? use mean over a few other functions at same j
        err_cross.append(np.mean([(u[m,j]-u[n,j])**2 for m in other[:20]]))
mse_same=float(np.mean(err_same)); mse_cross=float(np.mean(err_cross))
ratio=mse_same/mse_cross

# figures
plt.figure(figsize=(6.4,4.2))
plt.plot(x,a[0],label='a(x) sample')
plt.plot(x,u[0]*10,label='u(x) x10')
plt.xlabel('x'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题1Darcy样本.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
plt.hist(same_d,bins=np.arange(0,12)-0.5,rwidth=0.8)
plt.xlabel('grid distance to nearest same-function training point')
plt.ylabel('count'); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题1距离分布.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
plt.plot(np.array(hL)[:,0],np.array(hL)[:,1],label='S_point train')
plt.plot(np.array(hG)[:,0],np.array(hG)[:,1],label='S_func train')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('MSE (normalized)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题1训练曲线.png"),dpi=180); plt.close()

fig,ax=plt.subplots(figsize=(7.2,4.2))
labels=['L leak pts','L unseen','G leak funcs','G unseen']
vals=[mse_leak,mse_unseenL,mse_leakG,mse_unseenG]
ax.bar(labels,vals,color=['C0','C0','C1','C1'],alpha=0.7)
ax.set_yscale('log'); ax.set_ylabel('MSE'); 
for i,v in enumerate(vals): ax.text(i,v,f'{v:.3e}',ha='center',va='bottom',fontsize=8)
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题1MSE对比.png"),dpi=180); plt.close()

res={
 "mse_leak":mse_leak,"rel_leak":rel_leak,
 "mse_unseenL":mse_unseenL,"rel_unseenL":rel_unseenL,
 "mse_leakG":mse_leakG,"rel_leakG":rel_leakG,
 "mse_unseenG":mse_unseenG,"rel_unseenG":rel_unseenG,
 "ratio_mse_L":mse_leak/mse_unseenL,
 "ratio_mse_G":mse_leakG/mse_unseenG,
 "leak_rate":leak_rate,"frac_dist_le2":frac2,
 "one_nn_same":mse_same,"one_nn_cross":mse_cross,"one_nn_ratio":ratio,
 "time_sec":time.time()-t0
}
with open(os.path.join(ROOT,"A1组-6.3-习题1结果.json"),"w",encoding="utf-8") as f:
    json.dump(res,f,ensure_ascii=False,indent=2)
print(json.dumps(res,ensure_ascii=False,indent=2))




{
  "mse_leak": 8.249463761955428e-06,
  "rel_leak": 0.023815577682283425,
  "mse_unseenL": 0.006528681432246571,
  "rel_unseenL": 0.5838231720949991,
  "mse_leakG": 0.005137977539064351,
  "rel_leakG": 0.5946143577816068,
  "mse_unseenG": 0.005415076660888288,
  "rel_unseenG": 0.531705183336471,
  "ratio_mse_L": 0.001263572721010638,
  "ratio_mse_G": 0.9488282181071686,
  "leak_rate": 1.0,
  "frac_dist_le2": 0.9993589743589744,
  "one_nn_same": 6.417129201903948e-05,
  "one_nn_cross": 0.004508988884001686,
  "one_nn_ratio": 0.014231858554081872,
  "time_sec": 64.44122052192688
}


### 6.3-1 结果图

![6.3-1 Darcy 样本](图片/A1组-6.3-习题1Darcy样本.png)

![6.3-1 距离分布](图片/A1组-6.3-习题1距离分布.png)

![6.3-1 训练曲线](图片/A1组-6.3-习题1训练曲线.png)

![6.3-1 MSE 对比](图片/A1组-6.3-习题1MSE对比.png)

## 题 6.3-2 协议 S 与协议 O：同分布拟合不等于跨参数泛化

对应题面 (1)–(4)。本单元完成：协议 S（只用同一 $\mu_0$ 的训练数据）与协议 O（多个参数组合训练、留出未见组合）的 DeepONet 训练；15 个未见参数组合的逐组误差。重点结论：协议 S 在同分布上拟合很好，但跨参数预测明显错误；只有协议 O 才检验算子对参数组合的泛化。

输出：`数据/A1组-6.3-习题2结果.json`；图见 `图片/A1组-6.3-习题2*.png`。

In [4]:
import os, json, math, time
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)
np.random.seed(3); torch.manual_seed(3)
ROOT = "数据"
FIG = "图片"
assert os.path.isdir(FIG), "缺少 图片/ 目录；请在提交包根目录下启动 Jupyter"
T=0.3
xg=np.linspace(0,1,64)

def exact_u(x,t,A,B,c,D):
    return (A*np.exp(-4*np.pi**2*D*t)*np.sin(2*np.pi*(x-c*t))
            +B*np.exp(-16*np.pi**2*D*t)*np.sin(4*np.pi*(x-c*t)))

def u0_grid(A,B):
    return A*np.sin(2*np.pi*xg)+B*np.sin(4*np.pi*xg)

class MLP(nn.Module):
    def __init__(self,indim,width=64,depth=2,outdim=64):
        super().__init__()
        layers=[nn.Linear(indim,width),nn.Tanh()]
        for _ in range(depth-1): layers += [nn.Linear(width,width),nn.Tanh()]
        layers += [nn.Linear(width,outdim)]
        self.net=nn.Sequential(*layers)
        for p in self.parameters():
            if p.ndim==2: nn.init.xavier_normal_(p)
    def forward(self,x): return self.net(x)

class DeepONet(nn.Module):
    def __init__(self,p=256):
        super().__init__()
        self.branch=MLP(66,p,2,p)
        self.trunk=MLP(1,p,2,p)
        self.bias=nn.Parameter(torch.zeros(1))
    def forward(self,binputs,x):
        # binputs [B,66], x [N,1] -> [B,N]
        b=self.branch(binputs)
        t=self.trunk(x)
        return b @ t.T + self.bias

def make_params(n,rng):
    A=rng.uniform(0.5,1.5,n); B=rng.uniform(-0.8,0.8,n)
    c=rng.uniform(0.5,1.5,n); D=rng.uniform(0.01,0.05,n)
    return np.stack([A,B,c,D],axis=1)

# protocol S
mu0=np.array([1.0,0.4,0.9,0.02])
mu1=np.array([1.4,-0.6,1.2,0.04])
xS=np.linspace(0,1,512)
uS=exact_u(xS,T,*mu0)
rng=np.random.default_rng(11)
idx=rng.permutation(len(xS)); ntr=int(0.8*len(xS))
itr=idx[:ntr]; ite=idx[ntr:]
branchS=torch.tensor(np.concatenate([u0_grid(mu0[0],mu0[1]),mu0[2:]]).reshape(1,-1),dtype=torch.float32)
xS_t=torch.tensor(xS.reshape(-1,1),dtype=torch.float32)
uS_t=torch.tensor(uS.reshape(-1,1),dtype=torch.float32)
itr_t=torch.tensor(itr); ite_t=torch.tensor(ite)

def train_S(seed=0,epochs=30000,lr=1e-3):
    torch.manual_seed(seed)
    model=DeepONet()
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=1e-5)
    for ep in range(epochs):
        opt.zero_grad()
        pred=model(branchS,xS_t)[0]
        loss=((pred[itr_t]-uS_t[itr_t,0])**2).mean()
        loss.backward(); opt.step(); sched.step()
    with torch.no_grad():
        pred=model(branchS,xS_t)[0].numpy()
    rel_same=np.linalg.norm(pred[ite]-uS[ite])/np.linalg.norm(uS[ite])
    # cross mu1
    b1=torch.tensor(np.concatenate([u0_grid(mu1[0],mu1[1]),mu1[2:]]).reshape(1,-1),dtype=torch.float32)
    with torch.no_grad():
        pred1=model(b1,xS_t)[0].numpy()
    u1=exact_u(xS,T,*mu1)
    rel_cross=np.linalg.norm(pred1-u1)/np.linalg.norm(u1)
    return model,pred,rel_same,pred1,u1,rel_cross

# protocol O
rng2=np.random.default_rng(22)
params=make_params(100,rng2)
# ensure mu0 not in set? not necessary; split disjoint by index
perm=rng2.permutation(100)
tr=perm[:70]; va=perm[70:85]; te=perm[85:]
bO=np.stack([np.concatenate([u0_grid(p[0],p[1]),p[2:]]) for p in params])
yO=np.stack([exact_u(xg,T,*p) for p in params])
bO_t=torch.tensor(bO,dtype=torch.float32); yO_t=torch.tensor(yO,dtype=torch.float32)
xg_t=torch.tensor(xg.reshape(-1,1),dtype=torch.float32)
tr_t=torch.tensor(tr); te_t=torch.tensor(te)

def train_O(seed=0,epochs=15000,lr=1e-3):
    torch.manual_seed(seed)
    model=DeepONet()
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=1e-5)
    yscale=float(yO_t[tr_t].std())
    for ep in range(epochs):
        opt.zero_grad()
        pred=model(bO_t[tr_t],xg_t)
        loss=((pred-yO_t[tr_t]/yscale)**2).mean()
        loss.backward(); opt.step(); sched.step()
    with torch.no_grad():
        pred=model(bO_t[te_t],xg_t).numpy()*yscale
    true=yO[te]
    rels=np.linalg.norm(pred-true,axis=1)/np.linalg.norm(true,axis=1)
    # zero and mean baselines
    rel_zero=np.linalg.norm(true,axis=1)/np.linalg.norm(true,axis=1)
    mean_pred=np.mean(yO[tr],axis=0,keepdims=True)
    rel_mean=np.linalg.norm(mean_pred-true,axis=1)/np.linalg.norm(true,axis=1)
    return model,pred,true,rels,rel_zero,rel_mean,yscale

t0=time.time()
mS,predS,rel_same,pred1,u1,rel_cross=train_S(seed=0,epochs=30000)
print('S',rel_same,rel_cross)
mO,predO,trueO,relsO,rel_zero,rel_mean,yscale=train_O(seed=0,epochs=15000)
print('O',relsO.mean(),relsO.max(),np.std(relsO,ddof=1),rel_zero.mean(),rel_mean.mean())
print('time',time.time()-t0)

# branch std diagnostics
stdS=np.std(np.tile(np.array([mu0[0],mu0[1],mu0[2],mu0[3]]),(1,1)),axis=0)
stdO=np.std(params[tr],axis=0)

# figures
plt.figure(figsize=(7.2,4.2))
plt.plot(xS,uS,'k-',lw=2,label='true mu0')
plt.plot(xS,predS,'C0--',lw=1.5,label='S prediction mu0')
plt.plot(xS,u1,'C1-',lw=1.5,label='true mu1')
plt.plot(xS,pred1,'C1:',lw=1.5,label='S prediction mu1')
plt.xlabel('x'); plt.ylabel('u(x,T)'); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题2协议S.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.2))
plt.plot(xg,trueO[0],'k-',label='true O sample')
plt.plot(xg,predO[0],'C0--',label='DeepONet O')
plt.xlabel('x'); plt.ylabel('u(x,T)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题2协议O.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
plt.bar(np.arange(len(relsO)),relsO,color='C0')
plt.axhline(relsO.mean(),color='k',ls='--',label=f'mean={relsO.mean():.3f}')
plt.xlabel('unseen parameter set'); plt.ylabel(r'relative $L^2$'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.3-习题2逐组误差.png"),dpi=180); plt.close()

res={
 "S":{"rel_same":float(rel_same),"rel_cross":float(rel_cross),"ratio":float(rel_cross/rel_same)},
 "O":{"mean_rel":float(relsO.mean()),"std_rel":float(np.std(relsO,ddof=1)),"max_rel":float(relsO.max()),
      "zero_mean_rel":float(rel_zero.mean()),"mean_baseline_rel":float(rel_mean.mean())},
 "branch_std":{"S":[float(v) for v in stdS],"O":[float(v) for v in stdO]},
 "split":{"train":len(tr),"val":len(va),"test":len(te)},
 "time_sec":time.time()-t0
}
with open(os.path.join(ROOT,"A1组-6.3-习题2结果.json"),"w",encoding="utf-8") as f:
    json.dump(res,f,ensure_ascii=False,indent=2)
print(json.dumps(res,ensure_ascii=False,indent=2))






{
  "S": {
    "rel_same": 0.012201657804047583,
    "rel_cross": 1.2550523066799752,
    "ratio": 102.8591628150434
  },
  "O": {
    "mean_rel": 0.14083663732761528,
    "std_rel": 0.08829441922836422,
    "max_rel": 0.3232220142358288,
    "zero_mean_rel": 1.0,
    "mean_baseline_rel": 0.5365128887836248
  },
  "branch_std": {
    "S": [
      0.0,
      0.0,
      0.0,
      0.0
    ],
    "O": [
      0.27395039380181696,
      0.46912066044439144,
      0.2665119198201776,
      0.01237958984182777
    ]
  },
  "split": {
    "train": 70,
    "val": 15,
    "test": 15
  },
  "time_sec": 263.6937038898468
}


### 6.3-2 结果图

![6.3-2 协议 S](图片/A1组-6.3-习题2协议S.png)

![6.3-2 协议 O](图片/A1组-6.3-习题2协议O.png)

![6.3-2 逐组误差](图片/A1组-6.3-习题2逐组误差.png)

## 附录 6.3-1 离散 Darcy 残差检查

快速核对 160 个样本的离散残差与解的取值范围。该检查对应解答 6.3-1 中的数据核对段落，不训练网络。

In [5]:
import numpy as np, os, json
N=160; Nx=64
x=np.linspace(0,1,Nx); h=x[1]-x[0]
rng=np.random.default_rng(2026)
xi=rng.standard_normal((N,12)); eta=rng.standard_normal((N,12))
loga=np.zeros((N,Nx))
for k in range(12):
    loga += (0.8/(k+1))*(xi[:,k,None]*np.cos(2*np.pi*(k+1)*x)[None,:]
                         + eta[:,k,None]*np.sin(2*np.pi*(k+1)*x)[None,:])
loga=np.clip(loga,-1.5,1.5); a=np.exp(loga)
u=np.zeros((N,Nx)); res=[]
for n in range(N):
    af=2*a[n,:-1]*a[n,1:]/(a[n,:-1]+a[n,1:]); m=Nx-2
    A=np.zeros((m,m)); rhs=np.full(m,h*h)
    for j in range(1,Nx-1):
        row=j-1; left=af[j-1]; right=af[j]
        if row-1>=0: A[row,row-1]=-left
        A[row,row]=left+right
        if row+1<m: A[row,row+1]=-right
    u[n,1:-1]=np.linalg.solve(A,rhs)
    # discrete residual interior: -(a_{j+1/2}(u_{j+1}-u_j)-a_{j-1/2}(u_j-u_{j-1}))/h^2 - 1
    r=-(af[1:]*(u[n,2:]-u[n,1:-1])-af[:-1]*(u[n,1:-1]-u[n,:-2]))/h**2 - 1.0
    res.append(r)
res=np.array(res)
print('max_abs',np.max(np.abs(res)),'rms',np.sqrt(np.mean(res**2)))
print('u min/max/std',u.min(),u.max(),u.std())


max_abs 2.2639667918156192e-12 rms 1.6966394208111074e-13
u min/max/std 0.0 0.3387828143308615 0.06947155565481364
